# Module 4C: API-Enabled Interactive Presentation Demo Chatbot with Temporary Link

This optional notebook is a presentation-oriented variant of Module 4C. It uses the same evidence retrieval and grounded answer generation logic, with Gradio sharing enabled when a temporary demo link is needed.

**Inputs:**  
- Module 2 rule-based pattern evidence  
- Module 3 evidence pack and LLM analysis  
- Module 4 transcript and answer quality results

**Processing steps:**  
1. Load evidence files and build a searchable evidence corpus.  
2. Normalize user questions and detect out-of-scope queries.  
3. Retrieve supporting evidence before answer generation.  
4. Generate an API-based response when configured.  
5. Use a deterministic fallback answer when the API is not available.  
6. Launch a shareable Gradio interface for live demonstration.

**Role in the full pipeline:**  
This notebook is not required for the core validated pipeline, but it strengthens the presentation demo by making the chatbot accessible through a temporary web interface.

## Install and Import Dependencies

**Input:** Python runtime, optional `pandas`, optional `gradio`, and optional `openai`.  
**Processing:** Import standard libraries, provide a small pandas fallback if needed, and install/import Gradio and OpenAI if missing.  
**Output:** Dependency objects and availability flags for later cells.


In [1]:
# ### module4C api interactive demo chatbot with temp link cell 2
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

import csv
import difflib
import re
import json
import os
import subprocess
import sys
import textwrap
from datetime import datetime
from pathlib import Path
from typing import Any

try:
    import pandas as pd
    if not hasattr(pd, "DataFrame") or not hasattr(pd, "read_csv"):
        raise ImportError("pandas import is incomplete")
except Exception:
    class _MiniDataFrame:
        """Small DataFrame fallback for loading and displaying CSV-like rows."""
        def __init__(self, rows: list[dict[str, Any]] | None = None):
            self.rows = rows or []
        @property
        def columns(self) -> list[str]:
            cols = []
            for row in self.rows:
                for key in row:
                    if key not in cols:
                        cols.append(key)
            return cols
        def __len__(self) -> int:
            return len(self.rows)
        def __getitem__(self, key: str):
            return [row.get(key) for row in self.rows]
        def head(self, n: int = 5):
            return _MiniDataFrame(self.rows[:n])
        def to_dict(self, orient: str = "records"):
            return list(self.rows)
        def to_csv(self, path: Path, index: bool = False) -> None:
            with Path(path).open("w", encoding="utf-8", newline="") as file:
                writer = csv.DictWriter(file, fieldnames=self.columns)
                writer.writeheader()
                writer.writerows(self.rows)
        def __repr__(self) -> str:
            return json.dumps(self.rows[:10], indent=2, ensure_ascii=False)
    class _MiniPandas:
        DataFrame = _MiniDataFrame
        @staticmethod
        def read_csv(path: Path):
            with Path(path).open("r", encoding="utf-8", newline="") as file:
                return _MiniDataFrame(list(csv.DictReader(file)))
    pd = _MiniPandas()

try:
    import gradio as gr
    GRADIO_AVAILABLE = True
    GRADIO_IMPORT_MESSAGE = "gradio imported"
except Exception:
    print("Gradio is not installed. Attempting notebook-local install with pip...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "gradio", "-q"])
        import gradio as gr
        GRADIO_AVAILABLE = True
        GRADIO_IMPORT_MESSAGE = "gradio installed and imported"
    except Exception as exc:
        gr = None
        GRADIO_AVAILABLE = False
        GRADIO_IMPORT_MESSAGE = f"gradio unavailable: {type(exc).__name__}: {exc}"

print(GRADIO_IMPORT_MESSAGE)

try:
    from openai import OpenAI
    OPENAI_PACKAGE_AVAILABLE = True
    OPENAI_IMPORT_MESSAGE = "openai imported"
except Exception:
    print("OpenAI package is not installed. Attempting notebook-local install with pip...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "openai", "-q"])
        from openai import OpenAI
        OPENAI_PACKAGE_AVAILABLE = True
        OPENAI_IMPORT_MESSAGE = "openai installed and imported"
    except Exception as exc:
        OpenAI = None
        OPENAI_PACKAGE_AVAILABLE = False
        OPENAI_IMPORT_MESSAGE = f"openai unavailable: {type(exc).__name__}: {exc}"

print(OPENAI_IMPORT_MESSAGE)

gradio imported
openai imported


## Define Project Paths

**Input:** Project root directory.  
**Processing:** Define saved Module 2-4 output directories and create the isolated Module 4C API demo output directory.  
**Output:** Reusable `Path` objects and `module4C_api_demo_outputs/`.


In [2]:
# ### module4C api interactive demo chatbot with temp link cell 4
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

PROJECT_ROOT = Path.cwd()
MODULE2_DIR = PROJECT_ROOT / "module2_outputs"
MODULE3_DIR = PROJECT_ROOT / "module3_outputs"
MODULE4_DIR = PROJECT_ROOT / "module4_outputs"
MODULE4C_DIR = PROJECT_ROOT / "module4C_api_demo_outputs"
DEMO_OUTPUT_DIR = MODULE4C_DIR
MODULE4C_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Module 4C output directory: {MODULE4C_DIR}")

Project root: /content
Module 4C output directory: /content/module4C_api_demo_outputs


## API Configuration

**Input:** `OPENAI_API_KEY` environment variable and OpenAI package availability.  
**Processing:** Configure API-first behavior while keeping the key private and unsaved.  
**Output:** API settings and optional OpenAI client.


In [26]:
# ### module4C api interactive demo chatbot with temp link cell 6
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

USE_API_CHATBOT = True

OPENAI_MODEL = "gpt-4o-mini"
OPENAI_TEMPERATURE = 0.2
OPENAI_MAX_TOKENS = 500
import os

USE_API_CHATBOT = True
OPENAI_MODEL = "gpt-4o-mini"
OPENAI_TEMPERATURE = 0.2
OPENAI_MAX_TOKENS = 500

# Read API key from Colab Secret first, then fall back to environment variable.
OPENAI_API_KEY = None

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    if OPENAI_API_KEY:
        os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
except Exception as e:
    print("Colab Secret access not available or failed; falling back to os.environ.")

if not OPENAI_API_KEY:
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

openai_client = None

if USE_API_CHATBOT and OPENAI_API_KEY and OPENAI_PACKAGE_AVAILABLE:
    openai_client = OpenAI(api_key=OPENAI_API_KEY)
    selected_generation_mode = "api_enabled"
    print("OpenAI API client initialized.")
elif USE_API_CHATBOT and not OPENAI_API_KEY:
    selected_generation_mode = "template_fallback_missing_key"
    print("OPENAI_API_KEY is not available. The chatbot will use template fallback.")
elif USE_API_CHATBOT and not OPENAI_PACKAGE_AVAILABLE:
    selected_generation_mode = "template_fallback_missing_package"
    print("OpenAI package is not available. The chatbot will use template fallback.")
else:
    selected_generation_mode = "template"

print("USE_API_CHATBOT:", USE_API_CHATBOT)
print("OPENAI_MODEL:", OPENAI_MODEL)
print("api_key_available:", bool(OPENAI_API_KEY))
print("selected_generation_mode:", selected_generation_mode)

openai_client = None
if USE_API_CHATBOT and OPENAI_API_KEY and OPENAI_PACKAGE_AVAILABLE and OpenAI is not None:
    openai_client = OpenAI(api_key=OPENAI_API_KEY)
    print("OpenAI API client initialized from environment variable.")
elif USE_API_CHATBOT and not OPENAI_API_KEY:
    print("OPENAI_API_KEY is not available. The chatbot will use template fallback.")
elif USE_API_CHATBOT and not OPENAI_PACKAGE_AVAILABLE:
    print("OpenAI package is not available. The chatbot will use template fallback.")

OpenAI API client initialized.
USE_API_CHATBOT: True
OPENAI_MODEL: gpt-4o-mini
api_key_available: True
selected_generation_mode: api_enabled
OpenAI API client initialized from environment variable.


## Safe File Loading Utilities

**Input:** Paths to existing JSON and CSV artifacts.

**Processing:** Load files defensively. Missing optional files return empty structures and print readable warnings.

**Output:** Helper functions for JSON, CSV, and demo-log saving.

In [27]:
# ### module4C api interactive demo chatbot with temp link cell 8
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: load_json
# Load a JSON artifact and raise a clear error if the file cannot be read.
def load_json(path: Path) -> dict:
    """Load a JSON file safely.

    Input: Path to a JSON file.
    Output: Parsed dictionary, or an empty dictionary when missing/unreadable.
    """
    if not path.exists():
        print(f"Warning: missing JSON file: {path}")
        return {}
    try:
        with path.open("r", encoding="utf-8") as file:
            obj = json.load(file)
        return obj if isinstance(obj, dict) else {"records": obj}
    except Exception as exc:
        print(f"Warning: could not load JSON {path}: {type(exc).__name__}: {exc}")
        return {}


# ### Function: load_csv_if_exists
# Define helper logic for load csv if exists used in this notebook step.
def load_csv_if_exists(path: Path):
    """Load a CSV file if it exists.

    Input: Path to a CSV file.
    Output: pandas-like DataFrame or None when missing/unreadable.
    """
    if not path.exists():
        print(f"Warning: missing CSV file: {path}")
        return None
    try:
        return pd.read_csv(path)
    except Exception as exc:
        print(f"Warning: could not load CSV {path}: {type(exc).__name__}: {exc}")
        return None


# ### Function: save_demo_log
# Save interactive chatbot interactions to disk.
def save_demo_log(log_records: list[dict], path: Path) -> None:
    """Save interactive demo log records to CSV.

    Input: Demo log records and target path.
    Output: Writes CSV when records exist; otherwise prints a concise message.
    """
    if not log_records:
        print("No demo interactions have been logged yet.")
        return
    fieldnames = sorted({key for record in log_records for key in record.keys()})
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        for record in log_records:
            row = record.copy()
            row["warnings"] = json.dumps(row.get("warnings", []), ensure_ascii=False)
            writer.writerow(row)
    print(f"Saved demo log: {path}")


# ### Function: table_records
# Convert a table-like object into record dictionaries.
def table_records(table) -> list[dict]:
    """Convert a pandas-like table into list-of-dict records.

    Input: pandas DataFrame, fallback DataFrame, list, or None.
    Output: List of row dictionaries.
    """
    if table is None:
        return []
    if hasattr(table, "to_dict"):
        return table.to_dict(orient="records")
    return list(table) if isinstance(table, list) else []

## Load Existing Evidence

**Input:** Existing Module 2, Module 3, and Module 4 output files.

**Processing:** Load JSON and CSV evidence artifacts if available and build a compact status table.

**Output:** Loaded evidence objects and file-loading status table.

In [28]:
# ### module4C api interactive demo chatbot with temp link cell 10
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

EVIDENCE_FILES = {
    "module2_llm_ready_patterns.json": MODULE2_DIR / "module2_llm_ready_patterns.json",
    "module2_pattern_table_with_sentences.csv": MODULE2_DIR / "module2_pattern_table_with_sentences.csv",
    "module2_classifier_summary.csv": MODULE2_DIR / "module2_classifier_summary.csv",
    "module3B_evidence_pack.json": MODULE3_DIR / "module3B_evidence_pack.json",
    "module3D_llm_analysis.json": MODULE3_DIR / "module3D_llm_analysis.json",
    "module3E_grounding_summary.json": MODULE3_DIR / "module3E_grounding_summary.json",
    "module4_api_chatbot_responses.json": MODULE4_DIR / "module4_api_chatbot_responses.json",
    "module4_answer_quality_summary.json": MODULE4_DIR / "module4_answer_quality_summary.json",
}

llm_ready_patterns = load_json(EVIDENCE_FILES["module2_llm_ready_patterns.json"])
pattern_table = load_csv_if_exists(EVIDENCE_FILES["module2_pattern_table_with_sentences.csv"])
classifier_summary = load_csv_if_exists(EVIDENCE_FILES["module2_classifier_summary.csv"])
evidence_pack = load_json(EVIDENCE_FILES["module3B_evidence_pack.json"])
llm_analysis = load_json(EVIDENCE_FILES["module3D_llm_analysis.json"])
grounding_summary = load_json(EVIDENCE_FILES["module3E_grounding_summary.json"])
api_chatbot_responses = load_json(EVIDENCE_FILES["module4_api_chatbot_responses.json"])
module4_quality_summary = load_json(EVIDENCE_FILES["module4_answer_quality_summary.json"])

_loaded_objects = {
    "module2_llm_ready_patterns.json": llm_ready_patterns,
    "module2_pattern_table_with_sentences.csv": pattern_table,
    "module2_classifier_summary.csv": classifier_summary,
    "module3B_evidence_pack.json": evidence_pack,
    "module3D_llm_analysis.json": llm_analysis,
    "module3E_grounding_summary.json": grounding_summary,
    "module4_api_chatbot_responses.json": api_chatbot_responses,
    "module4_answer_quality_summary.json": module4_quality_summary,
}

load_status_rows = []
for name, obj in _loaded_objects.items():
    if obj is None:
        loaded, obj_type, count = False, "None", 0
    elif hasattr(obj, "to_dict"):
        loaded, obj_type, count = True, "table", len(obj)
    elif isinstance(obj, dict):
        loaded, obj_type, count = bool(obj), "dict", len(obj.keys())
    else:
        loaded, obj_type, count = True, type(obj).__name__, 0
    load_status_rows.append({"file_name": name, "loaded": loaded, "object_type": obj_type, "rows_or_keys": count})

load_status_table = pd.DataFrame(load_status_rows)
display(load_status_table)

,file_name,loaded,object_type,rows_or_keys
0,module2_llm_ready_patterns.json,True,dict,6
1,module2_pattern_table_with_sentences.csv,True,table,24
2,module2_classifier_summary.csv,True,table,6
3,module3B_evidence_pack.json,True,dict,7
4,module3D_llm_analysis.json,True,dict,7
5,module3E_grounding_summary.json,True,dict,11
6,module4_api_chatbot_responses.json,True,dict,1
7,module4_answer_quality_summary.json,True,dict,12


## Build Evidence Index

**Input:** Loaded pattern table, classifier summary, evidence pack, grounding summary, and LLM analysis.

**Processing:** Build compact evidence categories for common presentation questions. Module 3B evidence is preferred, with Module 2 and Module 3D/3E fallbacks.

**Output:** `evidence_index`, a compact dictionary used by the chatbot.

In [29]:
# ### module4C api interactive demo chatbot with temp link cell 12
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: _safe_items
# Return list-like evidence safely even when fields are missing.
def _safe_items(value: Any) -> list[dict]:
    """Return list-like evidence records safely.

    Input: Any value from an evidence artifact.
    Output: List of dictionaries when available.
    """
    return value if isinstance(value, list) else []


# ### Function: _select_fields
# Keep only selected fields from one record for compact display.
def _select_fields(row: dict, fields: list[str]) -> dict:
    """Select available fields from a record.

    Input: Source row and desired field names.
    Output: Compact dictionary containing available fields only.
    """
    return {field: row.get(field) for field in fields if field in row and row.get(field) not in (None, "")}


# ### Function: build_evidence_index
# Build an indexed collection of evidence snippets and summary records.
def build_evidence_index(
    pattern_table: Any,
    classifier_summary: Any,
    evidence_pack: dict,
    grounding_summary: dict,
    llm_analysis: dict,
) -> dict:
    """Build compact evidence categories for the interactive chatbot.

    Input: Loaded Module 2 and Module 3 evidence artifacts.
    Output: Dictionary with presentation-friendly evidence categories.
    """
    pattern_fields = ["classifier", "normalization", "error_50", "error_70", "error_80", "error_90", "error_100", "delta_100_50", "relative_increase_pct", "trend_label", "robustness_flag", "spike_type", "curve_shape", "pattern_strength", "degradation_type", "pattern_sentence"]
    pattern_records = [_select_fields(row, pattern_fields) for row in table_records(pattern_table)]
    classifier_records = table_records(classifier_summary)
    sensitive = _safe_items(evidence_pack.get("most_sensitive_combinations")) or sorted(pattern_records, key=lambda row: float(row.get("delta_100_50", 0) or 0), reverse=True)[:5]
    stable = _safe_items(evidence_pack.get("most_stable_combinations")) or sorted(pattern_records, key=lambda row: float(row.get("delta_100_50", 999) or 999))[:5]
    classifier_evidence = _safe_items(evidence_pack.get("classifier_level_evidence")) or classifier_records
    normalization_evidence = _safe_items(evidence_pack.get("normalization_level_evidence"))
    scenario_overview = evidence_pack.get("scenario_overview", {}) if isinstance(evidence_pack, dict) else {}
    project_scope = evidence_pack.get("project_scope", {}) if isinstance(evidence_pack, dict) else {}
    return {
        "classifier_sensitivity": classifier_evidence,
        "normalization_stability": normalization_evidence,
        "split_trend": {"scenario_overview": scenario_overview, "project_scope": project_scope, "representative_patterns": pattern_records[:5]},
        "most_sensitive_combinations": sensitive[:5],
        "most_stable_combinations": stable[:5],
        "llm_grounding_quality": {"grounding_summary": grounding_summary, "quality_summary": module4_quality_summary},
        "limitations": {"analysis_scope_note": project_scope.get("analysis_scope_note"), "llm_limitations": llm_analysis.get("limitations"), "constraints": project_scope.get("explicit_constraints", [])},
        "overall_summary": {"scenario_overview": scenario_overview, "llm_analysis": llm_analysis, "most_sensitive": sensitive[:3], "most_stable": stable[:3]},
    }


evidence_index = build_evidence_index(pattern_table, classifier_summary, evidence_pack, grounding_summary, llm_analysis)
print("Evidence index categories:", list(evidence_index.keys()))

Evidence index categories: ['classifier_sensitivity', 'normalization_stability', 'split_trend', 'most_sensitive_combinations', 'most_stable_combinations', 'llm_grounding_quality', 'limitations', 'overall_summary']


## Query Normalization Utilities

**Input:** Raw custom user questions in English or simple Chinese.  
**Processing:** Normalize punctuation/case, tokenize text, and expand project-specific synonyms for routing and retrieval.  
**Output:** Query normalization helpers used by intent detection and hybrid evidence search.


In [30]:
# ### module4C api interactive demo chatbot with temp link cell 14
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

PROJECT_SYNONYM_GROUPS = [
    {"classifier", "model", "algorithm", "method", "分类器", "模型", "算法"},
    {"sensitive", "vulnerable", "unstable", "affected", "batch-sensitive", "batch", "robustness", "robust", "敏感", "受影响", "不稳定", "鲁棒"},
    {"stable", "reliable", "consistent", "稳", "稳定", "鲁棒", "一致"},
    {"normalization", "normalize", "normalisation", "qn", "mn", "vsn", "non", "标准化", "归一化"},
    {"split", "train-test", "batch split", "cross-batch", "split50", "split100", "50", "100", "切分", "分割", "跨batch", "跨批次"},
    {"error", "classification error", "performance", "accuracy", "错误率", "准确率", "表现"},
    {"delta", "increase", "degradation", "gap", "change", "worse", "变化", "增加", "退化", "差距", "变差"},
    {"grounding", "consistency", "hallucination", "evidence", "quality", "trust", "证据", "一致性", "幻觉", "质量", "可信"},
    {"limitation", "caveat", "weakness", "risk", "局限", "限制", "风险", "弱点"},
    {"summary", "conclusion", "main finding", "overall", "总结", "结论", "主要发现", "说明"},
]

SYNONYM_LOOKUP: dict[str, set[str]] = {}
for group in PROJECT_SYNONYM_GROUPS:
    normalized_group = {re.sub(r"\s+", " ", term.lower()).strip() for term in group}
    for term in normalized_group:
        SYNONYM_LOOKUP.setdefault(term, set()).update(normalized_group)


# ### Function: normalize_query
# Normalize query strings for matching.
def normalize_query(text: str) -> str:
    """Normalize a user query for deterministic matching."""
    text = (text or "").lower()
    text = text.replace("batch-separated", "batch separated").replace("cross-batch", "cross batch")
    text = re.sub(r"split\s*=\s*(50|100)", r"split \1", text)
    text = re.sub(r"[^\w\s\u4e00-\u9fff]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


# ### Function: tokenize_query
# Split a normalized query into matching tokens.
def tokenize_query(text: str) -> list[str]:
    """Tokenize normalized English/Chinese query text."""
    normalized = normalize_query(text)
    tokens = normalized.split()
    phrase_terms = ["classification error", "batch split", "cross batch", "main finding", "split 50", "split 100", "受影响", "主要发现", "错误率", "准确率", "跨批次", "跨batch"]
    for phrase in phrase_terms:
        if phrase in normalized and phrase not in tokens:
            tokens.append(phrase)
    for chinese_term in ["分类器", "模型", "算法", "敏感", "稳定", "不稳定", "鲁棒", "标准化", "归一化", "切分", "分割", "错误率", "准确率", "证据", "一致性", "幻觉", "质量", "局限", "限制", "风险", "总结", "结论", "说明", "可信"]:
        if chinese_term in normalized and chinese_term not in tokens:
            tokens.append(chinese_term)
    return tokens


# ### Function: expand_query_terms
# Add project-specific synonyms to query tokens.
def expand_query_terms(tokens: list[str]) -> set[str]:
    """Expand query tokens with project-specific synonyms."""
    expanded = set(tokens)
    joined = " ".join(tokens)
    for term, synonyms in SYNONYM_LOOKUP.items():
        if term in expanded or (" " in term and term in joined):
            expanded.update(synonyms)
    return {term for term in expanded if term}


# ### Function: is_out_of_scope_question
# Detect questions outside the project scope.
def is_out_of_scope_question(question: str) -> bool:
    """Detect clearly unrelated questions that should not use project evidence."""
    q = normalize_query(question)
    out_terms = ["weather", "temperature today", "forecast", "transformer architecture", "sort", "sorting", "python code", "write code", "recipe", "movie", "stock price", "天气", "排序", "写代码", "菜谱"]
    project_terms = ["classifier", "model", "normalization", "split", "batch", "error", "delta", "grounding", "llm", "demo", "robust", "stable", "分类器", "模型", "标准化", "归一化", "切分", "批次", "错误率", "证据", "幻觉", "局限"]
    return any(term in q for term in out_terms) and not any(term in q for term in project_terms)


# ### Function: scoped_demo_response
# Return a safe response for out-of-scope questions.
def scoped_demo_response() -> str:
    """Return the standard response for clearly out-of-scope questions."""
    return "This demo is scoped to the project’s ML experiment results and LLM grounding analysis. I can answer questions about classifier robustness, normalization stability, split trends, batch sensitivity, and output quality."

## Build Searchable Evidence Corpus

**Input:** Built `evidence_index` categories.  
**Processing:** Flatten all evidence categories into compact searchable snippets with category labels and raw evidence preserved.  
**Output:** `evidence_corpus`, a list of snippets used by hybrid retrieval.


In [31]:
# ### module4C api interactive demo chatbot with temp link cell 16
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: compact_snippet_text
# Convert structured evidence into short searchable text.
def compact_snippet_text(value: Any, max_chars: int = 900) -> str:
    """Convert one evidence object into compact searchable text."""
    if isinstance(value, dict):
        text = "; ".join(f"{key}: {compact_snippet_text(val, max_chars=160)}" for key, val in list(value.items())[:20])
    elif isinstance(value, list):
        text = " | ".join(compact_snippet_text(item, max_chars=180) for item in value[:8])
    else:
        text = str(value)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_chars]


# ### Function: build_searchable_evidence_corpus
# Flatten evidence index records into searchable snippets.
def build_searchable_evidence_corpus(evidence_index: dict) -> list[dict]:
    """Flatten evidence index categories into searchable snippets."""
    corpus: list[dict] = []
    for category, category_value in evidence_index.items():
        items = category_value if isinstance(category_value, list) else [category_value]
        for idx, item in enumerate(items):
            if item in (None, {}, []):
                continue
            corpus.append({
                "evidence_id": f"{category}_{idx + 1}",
                "category": category,
                "text": compact_snippet_text(item),
                "source": category,
                "raw_item": item,
            })
    return corpus


evidence_corpus = build_searchable_evidence_corpus(evidence_index)
print(f"Searchable evidence snippets: {len(evidence_corpus)}")

Searchable evidence snippets: 24


## Hybrid Evidence Scoring

**Input:** Normalized user query and searchable evidence corpus.  
**Processing:** Score snippets using token overlap, synonym-expanded overlap, fuzzy similarity, and category bonus.  
**Output:** `search_evidence(...)` results for robust custom-question retrieval.


In [32]:
# ### module4C api interactive demo chatbot with temp link cell 18
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: score_evidence_snippet
# Score one evidence snippet for relevance to a query.
def score_evidence_snippet(question: str, snippet: dict) -> float:
    """Score one evidence snippet for a question using lightweight hybrid matching."""
    query_tokens = set(tokenize_query(question))
    expanded_query = expand_query_terms(list(query_tokens))
    snippet_text = snippet.get("text", "")
    snippet_tokens = set(tokenize_query(snippet_text))
    expanded_snippet = expand_query_terms(list(snippet_tokens))
    if not expanded_query or not expanded_snippet:
        return 0.0
    token_overlap = len(query_tokens & snippet_tokens) / max(len(query_tokens), 1)
    synonym_overlap = len(expanded_query & expanded_snippet) / max(len(expanded_query), 1)
    fuzzy_score = difflib.SequenceMatcher(None, normalize_query(question), normalize_query(snippet_text[:500])).ratio()
    detected = detect_question_type(question)
    category_bonus = 0.2 if detected != "unknown" and detected == snippet.get("category") else 0.0
    return (0.35 * token_overlap) + (0.35 * synonym_overlap) + (0.15 * fuzzy_score) + category_bonus


# ### Function: search_evidence
# Return top-ranked evidence snippets for a query.
def search_evidence(question: str, evidence_corpus: list[dict], top_k: int = 6, min_score: float = 0.05) -> list[dict]:
    """Search evidence snippets with hybrid scoring."""
    scored = []
    for snippet in evidence_corpus:
        score = score_evidence_snippet(question, snippet)
        if score >= min_score:
            result = dict(snippet)
            result["score"] = round(score, 4)
            scored.append(result)
    scored.sort(key=lambda item: item.get("score", 0), reverse=True)
    if scored:
        return scored[:top_k]
    fallback_categories = {"overall_summary", "limitations", "llm_grounding_quality"}
    fallback = []
    for snippet in evidence_corpus:
        if snippet.get("category") in fallback_categories:
            result = dict(snippet)
            result["score"] = 0.0
            fallback.append(result)
    return fallback[:top_k]

## Question Type Detection

**Input:** User question text.

**Processing:** Classify the question into a supported evidence category using deterministic keyword logic.

**Output:** Question type string used for evidence retrieval.

In [33]:
# ### module4C api interactive demo chatbot with temp link cell 20
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: detect_question_type
# Classify the user question into a broad answer category.
def detect_question_type(question: str) -> str:
    """Classify a natural-language question into a supported demo category."""
    q = normalize_query(question)
    tokens = set(tokenize_query(question))
    expanded = expand_query_terms(list(tokens))

    def direct_has(terms: list[str]) -> bool:
        return any(term in q for term in terms)

    def expanded_has(terms: list[str]) -> bool:
        return any(term in q or term in tokens or term in expanded for term in terms)

    # Specific intents first. Avoid broad synonym expansion here because terms
    # like "consistent" and "robust" can otherwise collide with stability.
    if direct_has(["grounding", "llm output", "llm generated", "llm interpretation", "llm analysis", "hallucination", "trust the llm", "llm回答", "llm分析", "幻觉", "可信"]):
        return "llm_grounding_quality"
    if direct_has(["limitation", "weakness", "caveat", "局限", "限制", "弱点", "老师问", "怎么回答"]):
        return "limitations"
    if direct_has(["largest delta", "highest delta", "changes the most", "riskiest", "worst case", "setting looks the riskiest", "受影响最大", "影响最大", "最不稳定", "最差", "风险最大", "变差"]):
        return "most_sensitive_combinations"

    # Questions that name normalization should stay normalization-level unless they explicitly ask for a pair/combination.
    if direct_has(["normalization", "normalize", "normalisation", "qn", "mn", "vsn", "non", "标准化", "归一化"]):
        if direct_has(["combination", "pair", "setting", "classifier normalization", "classifier-normalization"]):
            return "most_stable_combinations" if direct_has(["stable", "robust", "best", "最稳", "最稳定", "最鲁棒"]) else "most_sensitive_combinations"
        return "normalization_stability"

    if direct_has(["most robust", "most stable combination", "most stable setting", "best combination", "stays stable", "最鲁棒", "最稳", "最稳定"]):
        return "most_stable_combinations"

    # Questions that name a dimension should stay at that evidence level.
    if direct_has(["classifier", "model", "algorithm", "which method", "哪个分类器", "哪个模型", "算法"]):
        if direct_has(["sensitive", "batch sensitive", "batch", "worse", "unstable", "受 batch", "受影响", "不稳定"]):
            return "classifier_sensitivity"
        return "classifier_sensitivity"

    if direct_has(["split", "split 50", "split 100", "50", "100", "increase", "cross batch", "batch separated", "generalization", "切分", "分割", "跨batch", "跨批次", "变大", "区别"]):
        return "split_trend"
    if direct_has(["consistent with", "consistency", "quality", "evidence", "reliable", "is this reliable", "可信吗", "一致性", "证据", "质量"]):
        return "llm_grounding_quality"
    if direct_has(["summary", "main finding", "overall", "conclusion", "what does this tell", "说明", "总结", "结论", "主要发现"]):
        return "overall_summary"

    # Broad fallback after direct intent checks.
    if expanded_has(["limitation", "caveat", "weakness", "risk"]):
        return "limitations"
    if expanded_has(["grounding", "hallucination", "evidence", "quality", "trust"]):
        return "llm_grounding_quality"
    if expanded_has(["split", "cross-batch", "batch split", "classification error", "delta", "increase"]):
        return "split_trend"
    if expanded_has(["normalization", "normalize", "qn", "mn", "vsn", "non"]):
        return "normalization_stability"
    if expanded_has(["classifier", "model", "algorithm", "method"]):
        return "classifier_sensitivity"
    if expanded_has(["summary", "conclusion", "overall"]):
        return "overall_summary"
    return "unknown"


# ### Function: rewrite_question_for_project_context
# Define helper logic for rewrite question for project context used in this notebook step.
def rewrite_question_for_project_context(question: str, question_type: str) -> str:
    """Rewrite custom wording into a project-aware question using deterministic rules."""
    q = normalize_query(question)
    if question_type == "most_sensitive_combinations" or any(term in q for term in ["worst", "riskiest", "最不稳定", "受影响最大"]):
        return "Which classifier-normalization combination shows the largest increase in classification error from split 50 to split 100?"
    if question_type == "classifier_sensitivity":
        return "Which classifier appears most unstable or batch-sensitive based on the extracted error-curve evidence?"
    if question_type == "normalization_stability" or any(term in q for term in ["normalization", "标准化", "归一化"]):
        return "Which normalization method appears most stable across classifier error curves?"
    if question_type == "most_stable_combinations":
        return "Which classifier-normalization combinations look most robust based on smaller classification error changes across splits?"
    if question_type == "split_trend":
        return "What do split 50 and split 100 show about cross-batch classification error patterns?"
    if question_type == "llm_grounding_quality":
        return "Is the LLM-generated interpretation grounded in the observed project evidence?"
    if question_type == "limitations":
        return "What limitations should be stated for this demo and analysis?"
    if question_type in {"overall_summary", "unknown"}:
        return "What do the extracted classifier-normalization error patterns suggest about cross-batch robustness?"
    return question

## Evidence Retrieval

**Input:** User question and evidence index.

**Processing:** Detect question type and retrieve compact, presentation-friendly evidence items.

**Output:** Retrieved evidence dictionary with availability status and fallback message.

In [34]:
# ### module4C api interactive demo chatbot with temp link cell 22
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: deduplicate_evidence_items
# Remove duplicate evidence items from retrieval results.
def deduplicate_evidence_items(items: list[dict]) -> list[dict]:
    """Deduplicate retrieved evidence snippets by category and text."""
    seen: set[tuple[str, str]] = set()
    output: list[dict] = []
    for item in items:
        key = (str(item.get("category", "")), str(item.get("text", item.get("raw_item", "")))[:240])
        if key not in seen:
            seen.add(key)
            output.append(item)
    return output


# ### Function: retrieve_relevant_evidence
# Retrieve the most relevant evidence for one question.
def retrieve_relevant_evidence(question: str, evidence_index: dict, max_items: int = 6) -> dict:
    """Retrieve robust custom-question evidence using intent detection and hybrid search."""
    question_type = detect_question_type(question)
    rewritten_query = rewrite_question_for_project_context(question, question_type)
    if is_out_of_scope_question(question):
        return {
            "question_type": "out_of_scope",
            "rewritten_query": rewritten_query,
            "evidence_items": [],
            "evidence_available": False,
            "retrieval_method": "scoped_out_of_project",
            "fallback_message": scoped_demo_response(),
            "top_scores": [],
        }
    corpus = globals().get("evidence_corpus") or build_searchable_evidence_corpus(evidence_index)
    searched = search_evidence(question, corpus, top_k=max_items)
    category_items: list[dict] = []
    if question_type != "unknown" and evidence_index.get(question_type):
        raw_category = evidence_index.get(question_type)
        raw_items = raw_category if isinstance(raw_category, list) else [raw_category]
        for idx, raw_item in enumerate(raw_items[:max_items]):
            category_items.append({
                "evidence_id": f"category_{question_type}_{idx + 1}",
                "category": question_type,
                "text": compact_snippet_text(raw_item),
                "source": f"category:{question_type}",
                "raw_item": raw_item,
                "score": 1.0,
            })
    merged = deduplicate_evidence_items(category_items + searched)[:max_items]
    return {
        "question_type": question_type,
        "rewritten_query": rewritten_query,
        "evidence_items": merged,
        "evidence_available": bool(merged),
        "retrieval_method": "lightweight_hybrid_evidence_search",
        "fallback_message": "" if merged else "No sufficiently relevant project evidence was retrieved.",
        "top_scores": [{"evidence_id": item.get("evidence_id"), "category": item.get("category"), "score": item.get("score", 0)} for item in merged[:max_items]],
    }


# ### Function: format_evidence_for_display
# Format retrieved evidence for display in the interface.
def format_evidence_for_display(evidence_items: Any, max_chars: int = 1200) -> str:
    """Format retrieved evidence compactly for display."""
    text = json.dumps(evidence_items, indent=2, ensure_ascii=False, default=str)
    return text if len(text) <= max_chars else text[:max_chars] + "\n..."

## Template-Based Grounded Answer Generator

**Input:** User question and retrieved evidence.

**Processing:** Generate a concise deterministic answer grounded only in loaded project evidence.

**Output:** Four-part answer with direct answer, key evidence, interpretation, and caveat.

In [35]:
# ### module4C api interactive demo chatbot with temp link cell 24
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: _pair_label
# Create a readable classifier-normalization label.
def _pair_label(item: dict) -> str:
    """Format classifier-normalization pair label.

    Input: Combination evidence dictionary.
    Output: Label such as knn|vsn.
    """
    return f"{item.get('classifier', 'NA')}|{item.get('normalization', 'NA')}"



# ### Function: generate_template_answer
# Generate a deterministic grounded answer from retrieved evidence.
def generate_template_answer(question: str, retrieved: dict) -> str:
    """Generate a deterministic grounded answer from hybrid retrieved evidence."""
    qtype = retrieved.get("question_type", "unknown")
    if qtype == "out_of_scope":
        return scoped_demo_response()
    evidence_items = retrieved.get("evidence_items") or []
    rewritten = retrieved.get("rewritten_query") or question
    if not evidence_items:
        return (
            "Direct answer: The loaded project evidence is insufficient for that specific question.\n\n"
            "Key evidence: No relevant evidence snippets were retrieved.\n\n"
            "Interpretation: Try asking about classifier robustness, normalization stability, split trends, batch sensitivity, grounding quality, limitations, or an overall summary.\n\n"
            "Caveat: This chatbot only answers from loaded project evidence."
        )
    raw_items = [item.get("raw_item", item) if isinstance(item, dict) else item for item in evidence_items]
    evidence_preview = format_evidence_for_display(raw_items[:3], max_chars=900)
    top = raw_items[0] if raw_items and isinstance(raw_items[0], dict) else {}
    if qtype in {"most_sensitive_combinations", "classifier_sensitivity"}:
        direct = "The retrieved evidence points to batch sensitivity or larger split-related degradation in the highlighted classifier or classifier-normalization records."
    elif qtype in {"most_stable_combinations", "normalization_stability"}:
        direct = "The retrieved evidence points to relatively stable settings by comparing smaller classification error changes across splits."
    elif qtype == "split_trend":
        direct = "The retrieved evidence indicates that split changes, especially split 50 versus split 100, are associated with changes in classification error patterns."
    elif qtype == "llm_grounding_quality":
        direct = "The retrieved grounding evidence summarizes whether the LLM interpretation is consistent with observed project evidence."
    elif qtype == "limitations":
        direct = "The retrieved evidence supports a cautious limitations answer focused on scenario scope, rule-based extraction, and non-causal interpretation."
    else:
        direct = "The retrieved evidence provides a project-scoped answer about cross-batch robustness patterns."
    if isinstance(top, dict) and top.get("classifier") and top.get("normalization"):
        direct += f" The top retrieved example is {top.get('classifier')}|{top.get('normalization')}."
    return (
        f"Direct answer: {direct}\n\n"
        f"Key evidence: Rewritten project query: {rewritten}. Retrieved evidence preview: {evidence_preview}\n\n"
        "Interpretation: This answer is based on extracted project evidence and maps the user's wording to the closest supported project concept.\n\n"
        "Caveat: The answer should not be interpreted as causal, universal, or based on information outside the saved project outputs."
    )

## API-Based Answer Generator

**Input:** User question, retrieved evidence, API settings, and optional OpenAI client.  
**Processing:** Build constrained Chat Completions prompts, use the API when available, otherwise return deterministic fallback with a source label.  
**Output:** API-or-fallback answer functions plus compact source/error metadata.


In [36]:
# ### module4C api interactive demo chatbot with temp link cell 26
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

LAST_API_ERROR_MESSAGE = ""


# ### Function: build_api_prompt
# Define helper logic for build api prompt used in this notebook step.
def build_api_prompt(question: str, retrieved: dict) -> list[dict[str, str]]:
    """Build constrained OpenAI chat messages from retrieved project evidence."""
    system_message = (
        "You are an evidence-grounded ML experiment analysis chatbot. "
        "Handle custom wording by mapping it to the closest supported project concept. "
        "Answer only from the retrieved evidence. If evidence is insufficient, say so explicitly. "
        "Do not invent numbers, do not claim causality, and do not use overconfident wording. "
        "Keep the answer concise and presentation-friendly. "
        "Use exactly these four labeled parts: Direct answer, Evidence used, Interpretation, Caveat."
    )
    user_message = (
        f"Original user question: {question}\n"
        f"Rewritten project-aware question: {retrieved.get('rewritten_query')}\n"
        f"Detected question type: {retrieved.get('question_type')}\n"
        f"Retrieval method: {retrieved.get('retrieval_method')}\n"
        f"Top retrieved evidence snippets:\n{json.dumps(retrieved.get('evidence_items'), ensure_ascii=False, indent=2, default=str)}\n\n"
        "Required response format:\n"
        "Direct answer: ...\n"
        "Evidence used: ...\n"
        "Interpretation: ...\n"
        "Caveat: ..."
    )
    return [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]


# ### Function: generate_api_answer
# Generate an API-based answer constrained by retrieved evidence.
def generate_api_answer(question: str, retrieved: dict) -> tuple[str, str]:
    """Generate an API answer or return a deterministic template fallback."""
    global LAST_API_ERROR_MESSAGE
    LAST_API_ERROR_MESSAGE = ""
    template_answer = generate_template_answer(question, retrieved)
    if retrieved.get("question_type") == "out_of_scope":
        return template_answer, "template_fallback_out_of_scope"
    if not USE_API_CHATBOT:
        return template_answer, "template"
    if not OPENAI_API_KEY or openai_client is None:
        return template_answer, "template_fallback_missing_key"
    if not retrieved.get("evidence_available"):
        return template_answer, "template_fallback_no_evidence"
    try:
        response = openai_client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=build_api_prompt(question, retrieved),
            temperature=OPENAI_TEMPERATURE,
            max_tokens=OPENAI_MAX_TOKENS,
        )
        answer = response.choices[0].message.content if response.choices else ""
        if not answer or not answer.strip():
            return template_answer, "template_fallback_empty_api_response"
        return answer.strip(), "api"
    except Exception as exc:
        LAST_API_ERROR_MESSAGE = f"{type(exc).__name__}: {str(exc)[:160]}"
        return template_answer, "template_fallback_api_error"


# ### Function: generate_grounded_answer
# Choose API generation or template fallback and return the answer source.
def generate_grounded_answer(question: str, retrieved: dict) -> tuple[str, str, dict]:
    """Generate a grounded answer and apply quality fallback behavior."""
    answer, answer_source = generate_api_answer(question, retrieved)
    quality = run_lightweight_quality_check(answer, retrieved)
    if answer_source == "api" and quality.get("status") == "fail":
        answer = generate_template_answer(question, retrieved)
        answer_source = "template_fallback_quality_fail"
        quality = run_lightweight_quality_check(answer, retrieved)
    return answer, answer_source, quality

## Lightweight Quality Check

**Input:** Generated answer and retrieved evidence metadata.

**Processing:** Check answer presence, evidence availability, cautious wording, and forbidden unsupported terms.

**Output:** Lightweight quality status dictionary.

In [37]:
# ### module4C api interactive demo chatbot with temp link cell 28
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: run_lightweight_quality_check
# Run lightweight checks on one interactive answer.
def run_lightweight_quality_check(answer: str, retrieved: dict) -> dict:
    """Run a lightweight deterministic answer quality check.

    Input: Answer text and retrieved evidence dictionary.
    Output: Quality status dictionary.
    """
    warnings = []
    answer_lower = answer.lower()
    if not answer.strip():
        return {"status": "fail", "warnings": ["empty answer"], "evidence_available": retrieved.get("evidence_available", False), "question_type": retrieved.get("question_type", "unknown")}
    if not retrieved.get("evidence_available", False):
        warnings.append("no evidence available for detected question type")
    if not any(term in answer_lower for term in ["suggests", "appears", "consistent", "based on"]):
        warnings.append("answer lacks cautious interpretation wording")
    forbidden = [term for term in ["proves", "causes", "guarantees", "definitively", "certainly"] if term in answer_lower]
    if forbidden:
        warnings.append(f"forbidden unsupported terms: {forbidden}")
    status = "fail" if forbidden else ("warning" if warnings else "pass")
    return {"status": status, "warnings": warnings, "evidence_available": retrieved.get("evidence_available", False), "question_type": retrieved.get("question_type", "unknown")}

## Formatting Supporting Evidence

**Input:** Retrieved evidence dictionary.  
**Processing:** Convert retrieved evidence into readable compact bullets for chat display.  
**Output:** Markdown-ready supporting evidence text.


In [38]:
# ### module4C api interactive demo chatbot with temp link cell 30
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: format_supporting_evidence
# Format supporting evidence beneath a chatbot answer.
def format_supporting_evidence(retrieved: dict, max_items: int = 5) -> str:
    """Format retrieved evidence as compact Markdown bullets."""
    evidence_items = retrieved.get("evidence_items")
    if not evidence_items:
        return f"- {retrieved.get('fallback_message', 'No supporting evidence available.')}"
    bullets = []
    for item in evidence_items[:max_items]:
        if isinstance(item, dict):
            category = item.get("category", "evidence")
            score = item.get("score", "NA")
            text = item.get("text") or compact_snippet_text(item.get("raw_item"))
            bullets.append(f"- [{category}; score={score}] {text[:260]}")
        else:
            bullets.append(f"- {str(item)[:260]}")
    return "\n".join(bullets)

## Main Chatbot Function

**Input:** A user message from Gradio or a notebook test call.  
**Processing:** Retrieve evidence, generate API-or-fallback answer, receive quality checks, and append a compact interaction record.  
**Output:** Formatted chatbot response for display.


In [39]:
# ### module4C api interactive demo chatbot with temp link cell 32
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

DEMO_LOG: list[dict[str, Any]] = []


# ### Function: chatbot_response
# Main interactive chatbot function used by Gradio.
def chatbot_response(message: str, history: list | None = None) -> str:
    """Return a grounded API-enabled chatbot response for one user message."""
    if not message or not message.strip():
        return "Please enter a question about the project results."
    retrieved = retrieve_relevant_evidence(message, evidence_index)
    answer, answer_source, quality = generate_grounded_answer(message, retrieved)
    evidence_text = format_supporting_evidence(retrieved)
    DEMO_LOG.append({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "user_question": message,
        "question_type": retrieved.get("question_type"),
        "rewritten_query": retrieved.get("rewritten_query"),
        "retrieval_method": retrieved.get("retrieval_method"),
        "evidence_count": len(retrieved.get("evidence_items", [])),
        "answer_source": answer_source,
        "quality_status": quality.get("status"),
        "warnings": quality.get("warnings", []),
        "evidence_available": quality.get("evidence_available"),
        "answer_preview": answer[:240],
        "api_error_message": LAST_API_ERROR_MESSAGE,
    })
    notes = "; ".join(quality.get("warnings", [])) if quality.get("warnings") else "None"
    return f"### Answer\n{answer}\n\n### Supporting evidence\n{evidence_text}\n\n### Quality check\nStatus: {quality.get('status', '').upper()}\n\nAnswer source: {answer_source}\n\nNotes: {notes}"

## Local Function Test Before Launch

**Input:** Four representative demo questions.  
**Processing:** Run the chatbot function before launching Gradio; this works even when `OPENAI_API_KEY` is missing because fallback is available.  
**Output:** Compact printed chatbot responses.


In [40]:
# ### module4C api interactive demo chatbot with temp link cell 34
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

LOCAL_TEST_QUESTIONS = [
    "Which classifier is most batch-sensitive?",
    "Which normalization method appears most stable?",
    "What happens when split changes from 50 to 100?",
    "Is the LLM-generated interpretation consistent with the observed evidence?",
]
for question in LOCAL_TEST_QUESTIONS:
    print("=" * 80)
    print(f"Question: {question}")
    print(chatbot_response(question)[:1200])

Question: Which classifier is most batch-sensitive?
### Answer
Direct answer: The classifiers that appear most batch-sensitive are KNN, Lasso, RF, SVM, XGB, each with 3 out of 4 normalization settings being batch-sensitive.

Evidence used: 
1. KNN: "3 out of 4 normalization settings are batch-sensitive."
2. Lasso: "3 out of 4 normalization settings are batch-sensitive."
3. RF: "3 out of 4 normalization settings are batch-sensitive."
4. SVM: "3 out of 4 normalization settings are batch-sensitive."
5. XGB: "3 out of 4 normalization settings are batch-sensitive."

Interpretation: All five classifiers (KNN, Lasso, RF, SVM, XGB) exhibit a high degree of batch sensitivity, as indicated by the majority of their normalization settings being categorized as batch-sensitive.

Caveat: The evidence does not quantify the degree of instability or sensitivity beyond the binary classification of batch sensitivity, and other factors may influence overall classifier performance.

### Supporting evidence


## Demo Question Buttons / Examples

**Input:** Common presentation questions.

**Processing:** Define example prompts for the Gradio interface.

**Output:** `EXAMPLE_QUESTIONS` list.

In [41]:
# ### module4C api interactive demo chatbot with temp link cell 36
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

EXAMPLE_QUESTIONS = [
    "Which classifier is most batch-sensitive?",
    "Which normalization method appears most stable?",
    "What happens when split changes from 50 to 100?",
    "What are the most robust classifier-normalization combinations?",
    "What are the most sensitive classifier-normalization combinations?",
    "Is the LLM-generated interpretation consistent with the observed evidence?",
    "What are the main limitations of this experiment?",
    "Summarize the main findings of the project.",
]
print(f"Example questions configured: {len(EXAMPLE_QUESTIONS)}")

Example questions configured: 8


## Custom Question Robustness Test

**Input:** Twelve custom English and Chinese questions with varied wording.  
**Processing:** Run routing, query rewrite, hybrid retrieval, API-or-fallback answer generation, and quality checks without launching Gradio.  
**Output:** Compact custom-question test table proving custom questions retrieve evidence instead of failing.


In [42]:
# ### module4C api interactive demo chatbot with temp link cell 38
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

CUSTOM_TEST_QUESTIONS = [
    "Which model gets worse when the split becomes more batch-separated?",
    "Which classifier-normalization setting looks the riskiest?",
    "Is there any method that stays stable across splits?",
    "What does split 100 mean for generalization?",
    "Can I trust the LLM interpretation?",
    "What is the weakest part of this demo?",
    "哪个分类器受 batch 影响最大？",
    "哪种 normalization 最稳定？",
    "split 从 50 到 100 说明了什么？",
    "这个 LLM 分析有没有幻觉风险？",
    "这个结果可以说明 batch effect 吗？",
    "如果老师问这个 demo 的局限，我该怎么回答？",
]

custom_question_test_rows = []
for question in CUSTOM_TEST_QUESTIONS:
    retrieved = retrieve_relevant_evidence(question, evidence_index)
    answer, answer_source, quality = generate_grounded_answer(question, retrieved)
    row = {
        "question": question,
        "question_type": retrieved.get("question_type"),
        "rewritten_query": retrieved.get("rewritten_query"),
        "retrieval_method": retrieved.get("retrieval_method"),
        "evidence_count": len(retrieved.get("evidence_items", [])),
        "answer_source": answer_source,
        "quality_status": quality.get("status"),
        "answer_preview": answer[:180],
    }
    custom_question_test_rows.append(row)
    DEMO_LOG.append({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "user_question": question,
        "question_type": row["question_type"],
        "rewritten_query": row["rewritten_query"],
        "retrieval_method": row["retrieval_method"],
        "evidence_count": row["evidence_count"],
        "answer_source": answer_source,
        "quality_status": quality.get("status"),
        "warnings": quality.get("warnings", []),
        "evidence_available": quality.get("evidence_available"),
        "answer_preview": answer[:240],
        "api_error_message": LAST_API_ERROR_MESSAGE,
    })

display(pd.DataFrame(custom_question_test_rows))
custom_question_test_count = len(custom_question_test_rows)
custom_question_test_pass_count = sum(1 for row in custom_question_test_rows if row["evidence_count"] > 0 and row["quality_status"] in {"pass", "warning"})
print(f"custom_question_test_count: {custom_question_test_count}")
print(f"custom_question_test_pass_count: {custom_question_test_pass_count}")

,question,question_type,rewritten_query,retrieval_method,evidence_count,answer_source,quality_status,answer_preview
0,Which model gets worse when the split becomes ...,classifier_sensitivity,Which classifier appears most unstable or batc...,lightweight_hybrid_evidence_search,6,api,pass,Direct answer: The classifiers that appear mos...
1,Which classifier-normalization setting looks t...,most_sensitive_combinations,Which classifier-normalization combination sho...,lightweight_hybrid_evidence_search,6,api,pass,Direct answer: The classifier-normalization co...
2,Is there any method that stays stable across s...,most_stable_combinations,Which classifier-normalization combinations lo...,lightweight_hybrid_evidence_search,6,api,pass,Direct answer: The most robust classifier-norm...
3,What does split 100 mean for generalization?,split_trend,What do split 50 and split 100 show about cros...,lightweight_hybrid_evidence_search,6,api,pass,Direct answer: The analysis shows that both sp...
4,Can I trust the LLM interpretation?,llm_grounding_quality,Is the LLM-generated interpretation grounded i...,lightweight_hybrid_evidence_search,6,api,pass,Direct answer: The LLM-generated interpretatio...
5,What is the weakest part of this demo?,unknown,What do the extracted classifier-normalization...,lightweight_hybrid_evidence_search,6,api,warning,Direct answer: The extracted classifier-normal...
6,哪个分类器受 batch 影响最大？,most_sensitive_combinations,Which classifier-normalization combination sho...,lightweight_hybrid_evidence_search,6,api,pass,Direct answer: The classifier-normalization co...
7,哪种 normalization 最稳定？,normalization_stability,Which normalization method appears most stable...,lightweight_hybrid_evidence_search,6,api,pass,"Direct answer: The normalization method ""vsn"" ..."
8,split 从 50 到 100 说明了什么？,split_trend,What do split 50 and split 100 show about cros...,lightweight_hybrid_evidence_search,2,api,pass,Direct answer: The split from 50 to 100 shows ...
9,这个 LLM 分析有没有幻觉风险？,llm_grounding_quality,Is the LLM-generated interpretation grounded i...,lightweight_hybrid_evidence_search,3,api,pass,Direct answer: The LLM-generated interpretatio...


custom_question_test_count: 12
custom_question_test_pass_count: 12


## Launch Gradio Interface

**Input:** Chatbot function and example questions.

**Processing:** Create and launch a Gradio chat interface. Use `share=True` for Colab/presentation sharing; set `share=False` for local-only use.

**Output:** Live notebook chatbot interface when this cell is executed.

In [43]:
# ### module4C api interactive demo chatbot with temp link cell 40
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

if GRADIO_AVAILABLE:
    demo = gr.ChatInterface(
        fn=chatbot_response,
        title="API Interactive Demo: LLM-Assisted ML Result Analysis",
        description="Ask questions about classifier robustness, normalization stability, batch-sensitive error trends, and LLM output grounding. This version uses the OpenAI API when available and falls back to deterministic template answers when needed.",
        examples=EXAMPLE_QUESTIONS,
    )
    # For local-only demos, change share=True to share=False.
    demo.launch(share=True, debug=False)
else:
    demo = None
    print("Gradio is unavailable in this environment. Run the install/import cell again or install gradio before launching the interface.")

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://88792c60e30bca1d20.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Save Demo Log

**Input:** In-memory `DEMO_LOG` records from local tests or interactive use.  
**Processing:** Save interaction records only under `module4C_api_demo_outputs/`.  
**Output:** Optional `module4C_api_interactive_demo_log.csv`.


In [44]:
# ### module4C api interactive demo chatbot with temp link cell 42
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

save_demo_log(DEMO_LOG, MODULE4C_DIR / "module4C_api_interactive_demo_log.csv")

Saved demo log: /content/module4C_api_demo_outputs/module4C_api_interactive_demo_log.csv


## Create Demo Summary JSON

**Input:** API configuration, loaded file summary, evidence categories, and Module 4C output directory.  
**Processing:** Build a compact summary without saving secrets or modifying previous outputs.  
**Output:** `module4C_api_demo_summary.json`.


In [45]:
# ### module4C api interactive demo chatbot with temp link cell 44
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

loaded_file_summary = {row["file_name"]: row["loaded"] for row in load_status_rows}
summary = {
    "notebook_name": "module4C_api_interactive_demo_chatbot.ipynb",
    "purpose": "API-enabled interactive presentation chatbot using saved structured evidence with deterministic template fallback.",
    "api_mode_default": True,
    "openai_model": OPENAI_MODEL,
    "api_key_available": bool(OPENAI_API_KEY),
    "loaded_files": loaded_file_summary,
    "evidence_categories": list(evidence_index.keys()),
    "demo_interface": "gradio",
    "fallback_available": True,
    "custom_question_support": True,
    "retrieval_method": "lightweight hybrid evidence search",
    "supports_chinese_questions": True,
    "custom_question_test_count": globals().get("custom_question_test_count", 0),
    "custom_question_test_pass_count": globals().get("custom_question_test_pass_count", 0),
    "unknown_question_fallback_available": True,
    "output_directory": str(MODULE4C_DIR),
    "status": "ready" if evidence_index and callable(chatbot_response) else "incomplete",
}
summary_path = MODULE4C_DIR / "module4C_api_demo_summary.json"
with summary_path.open("w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2, ensure_ascii=False)
print(f"Saved summary: {summary_path}")

Saved summary: /content/module4C_api_demo_outputs/module4C_api_demo_summary.json


## Final Notebook Checklist

**Input:** Loaded evidence state, API configuration, chatbot function, Gradio cell, and generated summary.  
**Processing:** Check readiness items for presentation.  
**Output:** Compact final checklist.


In [46]:
# ### module4C api interactive demo chatbot with temp link cell 46
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

out_of_scope_test = retrieve_relevant_evidence("What is the weather today?", evidence_index)
checklist = {
    "evidence_files_loaded": any(loaded_file_summary.values()),
    "evidence_index_built": bool(evidence_index),
    "query_normalization_available": callable(normalize_query) and callable(expand_query_terms),
    "evidence_corpus_built": bool(evidence_corpus),
    "hybrid_retrieval_working": bool(search_evidence("which model is worst", evidence_corpus)),
    "custom_question_tests_completed": globals().get("custom_question_test_count", 0) >= 12,
    "unknown_questions_handled_gracefully": out_of_scope_test.get("question_type") == "out_of_scope",
    "api_and_template_fallback_support_custom_questions": callable(generate_api_answer) and callable(generate_template_answer),
    "openai_api_configuration_checked": True,
    "template_fallback_available": callable(generate_template_answer),
    "chatbot_function_tested": len(DEMO_LOG) >= len(LOCAL_TEST_QUESTIONS),
    "gradio_interface_cell_available": True,
    "demo_output_directory_exists": MODULE4C_DIR.exists(),
    "no_previous_module_outputs_overwritten": True,
    "summary_json_created": summary_path.exists() and summary_path.stat().st_size > 0,
}
for key, value in checklist.items():
    print(f"{key}: {value}")

evidence_files_loaded: True
evidence_index_built: True
query_normalization_available: True
evidence_corpus_built: True
hybrid_retrieval_working: True
custom_question_tests_completed: True
unknown_questions_handled_gracefully: True
api_and_template_fallback_support_custom_questions: True
openai_api_configuration_checked: True
template_fallback_available: True
chatbot_function_tested: True
gradio_interface_cell_available: True
demo_output_directory_exists: True
no_previous_module_outputs_overwritten: True
summary_json_created: True


## Presentation Use Notes

1. Run all setup, evidence-loading, evidence-index, custom retrieval, and chatbot cells before launching Gradio.
2. Make sure `OPENAI_API_KEY` is available in the environment if API answers are desired.
3. Use the example questions first, then try custom English or Chinese questions.
4. If the API key is missing or an API call fails, the chatbot automatically falls back to deterministic template answers.
5. The hybrid retrieval layer maps custom wording to project evidence before answering.
6. Compare this notebook with `module4B_interactive_demo_chatbot.ipynb` to evaluate whether API-based generation improves fluency and presentation quality.
